In [28]:
import functools
import os
import sys
import json

from ultralytics import YOLO
import pandas as pd

sys.path.append("..")
import ultralytics_validator_wrapper
from tflite_inference import TFLiteInferer
from tflite_inference_input_adapter import TFLiteInferenceWithInputAdapter

pd.options.display.float_format = '{:.3f}'.format

%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
BASE_EXPERIMENT_DIR = "../results/batch1_10_clahe_batch11_margin_15_rq_yolov8n_wb_alif_box_01_dfl_01_2nd_e20"
BASE_WEIGHTS_DIR = os.path.join(BASE_EXPERIMENT_DIR, "weights")
WEIGHTS_PTH = os.path.join(BASE_WEIGHTS_DIR, "best.pt")

HOLDOUT_PTH = "/home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif/project_holdout.yaml"
VAL_PTH = "/home/klymy/data/cam4_revB/yolo_dataset/batch11_margin_15_wb_alif/project.yaml"
POC_PTH = "/home/klymy/projects/researches/people_detection_pg2/quality_reduce_comparison/cam4_poc_15/project_pg.yaml"

if os.path.exists(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json")):
    with open(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json"), "r") as f:
        data = json.load(f)
    CONF_TH = float(data["conf_th"])
    IOU_TH = float(data["iou_th"])
else:
    print("No inference_params.json found, using default thresholds.")
    CONF_TH = 0.35
    IOU_TH = 0.50

args = dict(
    task="pose",
    device=[0],
    batch=128,
    verbose=False,
    conf=CONF_TH,
    iou=IOU_TH,
    imgsz=320,
    model="val",
    rect=False,
    single_cls=False,
)

In [30]:
model = YOLO(WEIGHTS_PTH)

In [31]:
inferer = TFLiteInferer(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_saved_model/best_full_integer_quant.tflite"),
    head_pth=os.path.join(BASE_WEIGHTS_DIR, "head_saved_model/head_float32.tflite"),
    conf_th=CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

# inferer_adapter = TFLiteInferenceWithInputAdapter(
#     yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_saved_model/best_full_integer_quant.tflite"),
#     head_pth=os.path.join(BASE_WEIGHTS_DIR, "head_saved_model/head_float32.tflite"),
#     adapter_pth="../input_adapter/input_adapter_full_integer_quant.tflite",
#     conf_th=CONF_TH,
#     iou_th=IOU_TH,
#     imgsz=320,
# )

In [32]:
args["data"] = VAL_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

# ultralytics_validator_wrapper.validate_model(
#     args,
#     model,
#     functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_adapter, args),
#     False,
# )

val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch11_margin_15_wb_alif/labels/val.cache... 732 images, 70 backgrounds, 0 corrupt: 100%|██████████| 732/732 [00:00<?, ?it/s]


tst validation: 100%|██████████| 6/6 [00:29<00:00,  4.90s/it]


                   all        732       2137      0.993      0.987      0.993      0.858      0.994      0.988      0.993      0.993
   task    F1     P     R  MAP@50
0   box 0.990 0.993 0.987   0.993
1  pose 0.991 0.994 0.988   0.993


In [33]:
args["data"] = HOLDOUT_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

# ultralytics_validator_wrapper.validate_model(
#     args,
#     model,
#     functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_adapter, args),
#     False,
# )

val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]


tst validation: 100%|██████████| 2/2 [00:06<00:00,  3.15s/it]


                   all        148        409       0.67      0.467      0.584      0.359      0.804       0.56      0.704      0.679
   task    F1     P     R  MAP@50
0   box 0.550 0.670 0.467   0.584
1  pose 0.660 0.804 0.560   0.704


In [34]:
args["data"] = POC_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

# ultralytics_validator_wrapper.validate_model(
#     args,
#     model,
#     functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_adapter, args),
#     False,
# )

val: Scanning /home/klymy/projects/researches/people_detection_pg2/quality_reduce_comparison/cam4_poc_15/labels/val.cache... 191 images, 5 backgrounds, 0 corrupt: 100%|██████████| 191/191 [00:00<?, ?it/s]


tst validation: 100%|██████████| 2/2 [00:08<00:00,  4.09s/it]


                   all        191        417      0.615       0.52      0.596      0.313      0.732      0.667      0.734      0.698
   task    F1     P     R  MAP@50
0   box 0.564 0.615 0.520   0.596
1  pose 0.698 0.732 0.667   0.734
